# TwoWayFeedback — solved velocity feeds the magnetic convection

`MagneticCoupled` · ASN 28 validation form (TRUST-LiMe).



## Introduction

Validates the **two-way coupling** (keyword `vitesse_hydraulique`): the
transporting velocity of the magnetic equation is the *solved* NS velocity,
not a data field.  Same physical case as `Coupled_Feedback` — $\mathbf{b} = (y,-x)$,
$\eta = 0$, and the Lorentz force $\mathbf{J}\times\mathbf{b}/\mu_0 = -2(x,y)/\mu_0^2$
is a pure gradient, so the momentum balance is the hydrostatic rest
$\mathbf{u} \to 0$, $p = -(x^2+y^2)/\mu_0^2\rho$.

**One-way control** (`Coupled_Feedback`): $\mathbf{b}$ stays frozen exactly.
**Two-way (this form)**: the decaying initial flow
$\mathbf{u}_0 = (0.1, 0)$ advects $\mathbf{b}$ — $(\mathbf{u}\cdot\nabla)\mathbf{b} = (u_y, -u_x)$
— and the $\mathbf{b}$ probes must deviate from the frozen field.

## What is asserted

| # | Quantity | Bound |
|---|:--|:--|
| 1 | velocity decays to hydrostatic rest | $\max|\mathbf{u}|$ (last row) $< 10^{-3}$ |
| 2 | hydrostatic balance (control, as `Coupled_Feedback`) | $\Delta p = -0.60 \pm 5\,\%$ |
| 3 | two-way discriminator | $10^{-3} < \max_t \|\mathbf{b}-\mathbf{b}_0\| < 0.5$ |

No `induction_stretching` source here: the form isolates the convection
feedback; the $(\mathbf{b}\cdot\nabla)\mathbf{u}$ path is exercised by the
Hartmann two-way follow-up.



In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()



## Helpers: read a TRUST probe (`.son`) file


In [ ]:
import numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, sonde):
    """Return (time, values) for probe `sonde` of case `case`."""
    path = f"{BUILD}/{case}_{sonde}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

print("BUILD =", BUILD)



## Run

Runs on **2 processors** (`nbProcs=2` + `case.partition()`).


In [ ]:
import pathlib, shutil

from trustutils import run

NB_PROCS = 2

run.reset()
case = run.addCase(".", "twoway_feedback.data", nbProcs=NB_PROCS)
case.partition()
run.printCases()
run.runCases()

build = pathlib.Path(run.BUILD_DIRECTORY)
build.mkdir(parents=True, exist_ok=True)
for son in pathlib.Path("src").glob(f"{case.dataFileName_}_*.son"):
    shutil.copy2(son, build / son.name)



## Checks

### #1 — velocity decays to the hydrostatic rest


In [ ]:
t_u, u = read_son(case.dataFileName_, "SONDE_U")
# 2 probe points x 2 components on the last time row
u_last = np.abs(u[-1]).max()
print(f"max |u| at t={t_u[-1]:.2f}: {u_last:.3e}  (started from 0.1)")
assert u_last < 1e-3, f"u did not decay: max |u| = {u_last}"
print("OK: u decays to the Lorentz hydrostatic rest (as in Coupled_Feedback).")



### #2 — hydrostatic balance (control, as in Coupled_Feedback)


In [ ]:
_, p1 = read_son(case.dataFileName_, "SONDE_P1")
_, p2 = read_son(case.dataFileName_, "SONDE_P2")
dp = p2[-1] - p1[-1]
expected = -(0.65**2 + 0.65**2 - 0.35**2 - 0.35**2)
err = abs(dp - expected) / abs(expected)
print(f"dp = {dp:.6g}   (analytic {expected:.4g}, err {err:.3e})")
assert dp < 0 and err < 0.05, f"pressure balance off: dp = {dp}"
print("OK: hydrostatic balance holds in the two-way mode.")



### #3 — two-way discriminator: b deviates from the frozen field

With $\eta = 0$ and the transport velocity live, the transient flow advects
$\mathbf{b}$ away from $(y, -x)$ and the deviation persists (nothing diffuses
it back).  The one-way control (`Coupled_Feedback`) pins the opposite:
machine-precision frozen-ness.


In [ ]:
t_bx, bx = read_son(case.dataFileName_, "SONDE_BX")
_, by = read_son(case.dataFileName_, "SONDE_BY")
# b0 = (y, -x): at the probe (0.35, 0.35) -> (0.35, -0.35)
dev = np.hypot(bx - 0.35, by + 0.35)
i_max = int(np.argmax(dev))
print(f"deviation at t = 0.05, 0.25, 0.55: {dev[1]:.3e}, {dev[len(dev)//2]:.3e}, {dev[-1]:.3e}")
print(f"max deviation = {dev[i_max]:.4e} at t = {t_bx[i_max]:.2f}")
assert dev.max() > 1e-3, \
    f"b stayed frozen: the NS velocity does not feed the magnetic convection (max dev {dev.max():.2e})"
assert dev.max() < 0.5, f"b deviation out of sanity range: {dev.max()}"
print("OK: b is advected by the solved velocity — the two-way coupling is live.")



## Computer Performance


In [ ]:
run.tablePerf()

